# 03 · Regularización y clasificación de los 8 géneros

**Semana 3 de la planificación.** En este notebook:

1. Pasamos de 2 a 8 géneros con la estrategia **uno contra el resto**.
2. Estudiamos cómo influye λ (la fuerza de la regularización ridge) en el sobreajuste y el subajuste (**hipótesis H1**).
3. Elegimos el mejor λ con el conjunto de **validación** y analizamos los errores con una matriz de confusión.

**Uno contra el resto** (clase `UnoContraResto` en `src/regresion_logistica.py`): entrenamos 8 regresiones logísticas binarias, una por género ("¿es Rock o no?", "¿es Folk o no?"…). Para clasificar una canción, cada modelo da su probabilidad y nos quedamos con el género que tenga la más alta.

In [ ]:
import sys, json, time
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.datos import cargar_procesados
from src.regresion_logistica import UnoContraResto, exactitud

CARPETA_FIGURAS = "../figuras"

datos = cargar_procesados("../datos/procesados.npz")
X_train, y_train = datos["X_train"], datos["y_train"]
X_val, y_val = datos["X_val"], datos["y_val"]
GENEROS = datos["generos"]
print("Entrenamiento:", X_train.shape, " Validación:", X_val.shape)

## 1. Curva de exactitud frente a λ

Para cada valor de λ entrenamos el modelo **solo con el entrenamiento** y medimos la exactitud en entrenamiento y en validación. Probamos valores muy distintos (de 0,01 a 10.000), repartidos de forma logarítmica porque lo que importa es el orden de magnitud.

Cada valor de λ supone entrenar 8 modelos, así que esta celda tarda unos minutos.

In [ ]:
valores_lambda = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000, 3000, 10000]
resultados = []

for lam in valores_lambda:
    inicio = time.time()
    modelo = UnoContraResto(lam=lam, max_iter=50).ajustar(X_train, y_train)
    resultados.append({
        "λ": lam,
        "exactitud train": exactitud(y_train, modelo.predecir(X_train)),
        "exactitud val": exactitud(y_val, modelo.predecir(X_val)),
        "modelos que convergen": sum(m.convergio for m in modelo.modelos),
        "segundos": round(time.time() - inicio, 1),
    })
    print(f"λ = {lam:>7}: val = {resultados[-1]['exactitud val']:.3f}")

tabla_lambda = pd.DataFrame(resultados)
tabla_lambda.round(3)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.semilogx(tabla_lambda["λ"], tabla_lambda["exactitud train"], "o-", label="entrenamiento")
plt.semilogx(tabla_lambda["λ"], tabla_lambda["exactitud val"], "o-", label="validación")
plt.axhline(1 / 8, color="gray", linestyle="--", label="azar (12,5 %)")
plt.xlabel("λ (escala logarítmica)")
plt.ylabel("Exactitud")
plt.title("Regresión logística (uno contra el resto): efecto de la regularización")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/03_curva_lambda.png", dpi=150)
plt.show()

**Cómo leer la curva:**

- **λ pequeño (izquierda):** casi no hay regularización. La exactitud de entrenamiento es muy alta pero la de validación es menor: el modelo **sobreajusta** (memoriza el entrenamiento).
- **λ grande (derecha):** la penalización obliga a que todos los coeficientes sean casi 0, así que el modelo apenas puede usar las características. Bajan las dos curvas: **subajuste**.
- **El mejor λ** está en medio, donde la exactitud de validación es máxima.

Esta es la comprobación de la **hipótesis H1**. Fijaos también en la columna "modelos que convergen" de la tabla: con λ muy pequeño puede que alguno de los 8 modelos no converja, por el mismo motivo que en el notebook 02.

## 2. Modelo final con el mejor λ

In [ ]:
mejor = tabla_lambda.loc[tabla_lambda["exactitud val"].idxmax()]
MEJOR_LAMBDA = mejor["λ"]
print(f"Mejor λ: {MEJOR_LAMBDA}  ->  exactitud en validación: {mejor['exactitud val']:.3f}")

modelo_final = UnoContraResto(lam=MEJOR_LAMBDA, max_iter=50).ajustar(X_train, y_train)
pred_val = modelo_final.predecir(X_val)
print("Exactitud en validación:", round(exactitud(y_val, pred_val), 3))

## 3. Matriz de confusión

Cada fila es el género **real** y cada columna el género **predicho**. La diagonal son los aciertos y lo demás son confusiones. Como en validación hay exactamente 100 canciones por género, **cada número es directamente un porcentaje** de las canciones de esa fila.

In [ ]:
matriz = confusion_matrix(y_val, pred_val, labels=GENEROS)

fig, eje = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matriz, display_labels=GENEROS).plot(ax=eje, cmap="Blues", colorbar=False)
plt.xticks(rotation=40, ha="right")
eje.set_xlabel("Género predicho")
eje.set_ylabel("Género real")
eje.set_title(f"Regresión logística (λ = {MEJOR_LAMBDA}) · validación")
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/03_matriz_confusion_logistica.png", dpi=150)
plt.show()

In [ ]:
aciertos_por_genero = pd.Series(np.diag(matriz) / matriz.sum(axis=1), index=GENEROS)
aciertos_por_genero.sort_values(ascending=False).round(2)

**Qué mirar:** qué géneros se reconocen bien y cuáles no, y entre qué géneros se producen las confusiones más frecuentes (las casillas fuera de la diagonal con números altos). Es de esperar que géneros poco definidos, como *Experimental* o *International*, tengan menos aciertos, y que se confundan géneros musicalmente parecidos.

Guardamos los resultados para compararlos con k-NN en el notebook 04.

In [ ]:
resultado = {
    "mejor_lambda": float(MEJOR_LAMBDA),
    "exactitud_val": float(exactitud(y_val, pred_val)),
    "aciertos_por_genero": {g: float(v) for g, v in aciertos_por_genero.items()},
}
with open("../resultados/regresion_logistica.json", "w") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)
tabla_lambda.to_csv("../resultados/curva_lambda.csv", index=False)
print("Resultados guardados en la carpeta resultados/")

## Resumen

- Hemos extendido nuestra regresión logística a 8 géneros con la estrategia uno contra el resto.
- La curva frente a λ muestra el paso de sobreajuste a subajuste y nos permite elegir λ con el conjunto de validación (H1).
- La matriz de confusión indica qué géneros distingue bien el modelo y cuáles confunde.
- **El conjunto de test sigue sin usarse**: se reserva para la evaluación final (semana 5).